# P1: Universal-Failure Query Analysis

Reads the queries that score `llm_grade=0` across all four retrieval configs, both
reference-drafter models, and both generator scales (8B, 70B) — the "hard core" that no
config/generator combination gets right. Categorizes each into retrieval gap /
generation truncation / content error / ambiguous framing.

See the improvement plan's P1 entry for the full design, and the report notes' 2026-07-09
entry for the write-up of the finding this notebook produced.

## Step 1: Find queries that fail everywhere

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("../results")


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def universal_failures(correctness_rows: list[dict], config_key: str = "config") -> set[str]:
    """Queries where llm_grade==0 for every (config, reference_model) combination."""
    by_query: dict[str, dict[str, list[float]]] = {}
    for row in correctness_rows:
        if row.get("llm_grade") is None:
            continue
        by_query.setdefault(row["query"], {}).setdefault(row[config_key], []).append(row["llm_grade"])

    all_configs = {row[config_key] for row in correctness_rows}
    fails = set()
    for query, by_config in by_query.items():
        if set(by_config.keys()) != all_configs:
            continue  # incomplete data for this query, skip
        if all(all(g == 0 for g in grades) for grades in by_config.values()):
            fails.add(query)
    return fails


In [ ]:
correctness_8b = load_jsonl(RESULTS_DIR / "correctness_scores.jsonl")
correctness_70b = load_jsonl(RESULTS_DIR / "ragas_correctness_scores_70b.jsonl")

# 8B rows use "ref_model", 70B rows use "reference_model" - normalize
for r in correctness_8b:
    r["reference_model"] = r.get("ref_model", r.get("reference_model"))

fail_8b = universal_failures(correctness_8b)
fail_70b = universal_failures(correctness_70b)
fail_both = fail_8b & fail_70b

print(f"8B universal failures: {len(fail_8b)}")
print(f"70B universal failures: {len(fail_70b)}")
print(f"Fail at BOTH generator scales: {len(fail_both)}")


## Step 2: Pull full context for each failing query

For each of the 21, check whether the gold clause was ever retrieved by any config at either generator scale, and gather every generator's answer for later reading.

In [ ]:
answers_8b = load_jsonl(RESULTS_DIR / "answers_recovered.jsonl")
answers_70b = load_jsonl(RESULTS_DIR / "answers_70b_recovered.jsonl")
test_set = load_jsonl(Path("../data/test_set.jsonl"))
ts_lookup = {r["query"]: r for r in test_set}

a8b_by_query: dict[str, list[dict]] = {}
for r in answers_8b:
    a8b_by_query.setdefault(r["query"], []).append(r)
a70b_by_query: dict[str, list[dict]] = {}
for r in answers_70b:
    a70b_by_query.setdefault(r["query"], []).append(r)

out_path = RESULTS_DIR / "p1_universal_failures.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for i, query in enumerate(sorted(fail_both), start=1):
        ts_row = ts_lookup[query]
        gold_ids = set(ts_row["gold_ids"])

        rows_8b = a8b_by_query.get(query, [])
        rows_70b = a70b_by_query.get(query, [])

        # Was the gold clause ever retrieved, by any config, at either generator scale?
        # (retrieval doesn't depend on generator, so 8B/70B rows for the same config
        # should have identical `retrieved` lists - checking both anyway as a sanity
        # cross-check, not assuming.)
        retrieved_hit = {}
        for r in rows_8b + rows_70b:
            cfg = r["config"]
            hit = bool(gold_ids & set(r["retrieved"]))
            retrieved_hit.setdefault(cfg, []).append(hit)
        any_config_retrieved_gold = any(any(hits) for hits in retrieved_hit.values())

        record = {
            "n": i,
            "query": query,
            "query_type": ts_row["query_type"],
            "gold_ids": ts_row["gold_ids"],
            "gold_ever_retrieved": any_config_retrieved_gold,
            "retrieved_hit_by_config": retrieved_hit,
            "reference_72b": ts_row["reference_72b"],
            "answers_8b": {r["config"]: {"answer": r["answer"], "citations": r["citations"],
                                          "retrieved": r["retrieved"]} for r in rows_8b},
            "answers_70b": {r["config"]: {"answer": r["answer"], "citations": r["citations"]}
                            for r in rows_70b},
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(fail_both)} rows to {out_path}")
n_retrieval_gap = sum(
    1 for l in out_path.read_text(encoding="utf-8").splitlines()
    if l.strip() and not json.loads(l)["gold_ever_retrieved"]
)
print(f"Automated split: {n_retrieval_gap}/{len(fail_both)} never had the gold clause "
      f"retrieved by ANY config - {len(fail_both) - n_retrieval_gap}/{len(fail_both)} had "
      f"it retrieved somewhere but still failed (needs manual read to see why).")


## Step 3: Condensed triage view of all 21

In [ ]:
rows = load_jsonl(RESULTS_DIR / "p1_universal_failures.jsonl")
for r in rows:
    print(f"--- #{r['n']} [{r['query_type']}] gold_ever_retrieved={r['gold_ever_retrieved']} ---")
    print("Q:", r["query"][:110])
    print("gold_ids:", r["gold_ids"])
    hybrid8b = r["answers_8b"].get("hybrid", {})
    print("hybrid/8b answer:", (hybrid8b.get("answer") or "")[:200].replace(chr(10), " "))
    print()


## Step 4: Full reference comparison for the surprising `exact_anchor` cases

Several `exact_anchor` answers (normally the easier query type) looked plausible in the
triage view yet still scored 0 across every config/generator/reference. Reading the full
reference against the full answer for these specifically.

In [ ]:
for n in [14, 15, 17, 19, 21]:
    r = next(x for x in rows if x["n"] == n)
    print(f"=== #{n} [{r['query_type']}] ===")
    print("Q:", r["query"])
    print()
    print("REFERENCE (72b):", r["reference_72b"])
    print()
    print("hybrid/8b ANSWER:", r["answers_8b"]["hybrid"]["answer"])
    print()
    print("-" * 100)


**Finding**: these are not hallucinations. The generator correctly identifies and cites
the right clause and correctly states the *first* condition/requirement it enumerates -
then stops. #14 gives only the first half of a two-part timing rule; #19 gives only one
of four sub-requirements; #17 covers 2 of 3 alternative conditions. This is a
*truncation* pattern, not a synthesis or hallucination failure.

## Step 5: Check the judge's reasoning for #21

#21's answer restates all three enumerated conditions almost verbatim against the
reference, yet still scored 0. Checking whether this is a judge-strictness artifact or a
real (if subtle) gap.

In [ ]:
correctness_rows = load_jsonl(RESULTS_DIR / "correctness_scores.jsonl")
target = "What three conditions must all be satisfied for a person to commit the failure-to-disclose offence under section"
for r in correctness_rows:
    if r["query"].startswith(target) and r["config"] == "hybrid":
        print("ref_model:", r["ref_model"], " llm_grade:", r["llm_grade"])
        print("judge_reply:", r.get("judge_reply", "(none)"))
        print()


The correctness-grade prompt only captures a yes/no verdict, not reasoning - a real gap
in the current correctness-check design worth fixing if this kind of borderline case
needs auditing again. Re-reading #21 by hand: the answer never states the actual
disclosure-*failure* that makes the three conditions into an offense - it lists the
conditions but omits the clause's concluding operative sentence entirely. Same
truncation pattern as the others, just at the very end of the clause rather than after
the first item.

## Step 6: Remaining retrieved-but-failed rows

In [ ]:
for n in [1, 2, 3, 5, 8, 9, 10, 11, 12, 13, 16, 20]:
    r = next(x for x in rows if x["n"] == n)
    print(f"=== #{n} [{r['query_type']}] ===")
    print("Q:", r["query"][:100])
    print("REF:", r["reference_72b"][:300])
    print("ANS:", r["answers_8b"]["hybrid"]["answer"][:300])
    print()


## Step 7: The 4 genuine retrieval-gap queries

Check retrieved clause IDs directly against gold IDs, and look at what the model says
when it never had the right clause available.

In [ ]:
for r in rows:
    if not r["gold_ever_retrieved"]:
        print(f"=== #{r['n']} [{r['query_type']}] ===")
        print("Q:", r["query"][:110])
        print("gold_ids:", r["gold_ids"])
        print("hybrid retrieved:", r["answers_8b"]["hybrid"]["retrieved"])
        print("ANS:", r["answers_8b"]["hybrid"]["answer"][:200])
        print()


## Findings summary

Final categorization of the 21 universal-failure queries:

- **4/21 (19%) genuine retrieval gaps** — gold clause never in any config's top-10. 2 of
  4 involve `mlr_2017_reg_3` (MLR 2017's definitions section), suggesting a specific
  hard-to-retrieve clause, not random misses. All 4 generated answers are confidently
  wrong (including one citation, `s21A(2)`, that isn't a valid clause ID anywhere in
  this corpus) rather than a safe "context doesn't answer" refusal — the same
  compliance-risk pattern as the earlier POCA s328 case.
- **~12/21 (57%) systematic truncation — the dominant failure mode.** The generator
  cites the right clause, states the first 1–2 enumerated conditions correctly, then
  stops — dropping the rest, and especially the clause's concluding/operative sentence.
  Not hallucination: the stated content is accurate, just incomplete.
- **~4/21 (19%) genuine content errors** — wrong section cited despite the correct one
  being available elsewhere in the retrieved set, or a clause's own content flatly
  misstated.
- **~1/21** ambiguous framing mismatch, not a clear error either way.

**Implication**: only ~19% of the hardest queries are retrieval failures. Retrieval-side
tuning was already shown to be close to exhausted on this corpus (two negative
experiments); this confirms generation is the right next investment, and sharpens the
fix considerably — not a generic "help the model synthesize better" but specifically
"stop truncating multi-part clause content." See the improvement plan's P2 entry.